# Collected edits

All three dermatologists go into one codebook per model, one row per case. A case marked wrong by anyone is edited; if the corrections disagree it is printed and dropped. Reliability, text generality, and image generality run first. Locality is fourth: the cases nobody marked wrong. Overall is reliability and locality combined.

`dermedit` kernel. `OVERWRITE` recomputes saved runs. `RESUME` finishes a run that is still missing a metric. With both off, a complete result is printed from disk and an unfinished one stays blank.

In [ ]:
import sys
from pathlib import Path

PERTURB = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/perturb_generality")
sys.path.insert(0, str(PERTURB))
from utils.bootstrap import bootstrap, results_dir
bootstrap()

import config as C

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False
RESUME = True

# EMBED_MODE:
#   standalone_dual  DINOv2 image concatenated with SBERT text. This is the current run.
#   dual_sbert       VLM vision activation concatenated with SBERT. Still dual.
#   vision           VLM vision activation only.
#   language         A middle language-layer activation only.
#   language_last    Late language-layer activation. The old VLM-key style.
EMBED_MODE = "language_last"
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = results_dir(PERTURB, "collect", EMBED_MODE)
POOL = "all"
print("results:", OUT)

results: /scratch/jq2uw/MME/derm_vlm_edit/editing/perturb_generality/results/collect


In [ ]:
import gc

from IPython.display import Markdown, display

from utils.edit import edit
from utils.edits import collect_all, collect_locality
from utils.eval import load_suite, load_vlm, needs_compute, render_table, run_suite

rows = []
for model_short in MODELS:
    if not needs_compute(OUT, POOL, model_short, overwrite=OVERWRITE, resume=RESUME):
        print(f"{POOL} / {model_short}: loaded")
        rows.append(load_suite(OUT, POOL, model_short))
        continue

    edits, report = collect_all(
        USERS, model_short, rationale_mode=RATIONALE_MODE,
        norm_answer=NORM_ANSWER, subsample=SUBSAMPLE,
    )
    print(f"{model_short}: {len(edits)} unique edits, {len(report['conflicts'])} conflicting cases dropped")
    for row in report["conflicts"]:
        users = ", ".join(str(u).split("@")[0] for u in row["users"])
        print(f"  drop {row['case_id']}: {users} -> {', '.join(row['answers'])}")
    locality = collect_locality(USERS, model_short, subsample=SUBSAMPLE)
    print(f"{model_short}: {len(locality)} locality cases")
    if not edits:
        print(f"skip {model_short}")
        continue

    import torch
    vlm, config = load_vlm(
        model_short, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
        embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
        image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
        scaler_dataset=SCALER_DATASET,
    )
    editor, queries = edit(vlm, config, edits)
    rows.append(run_suite(
        editor, vlm, queries, user=POOL, model_short=model_short,
        out_dir=OUT, locality_edits=locality, seed=SEED,
        norm_answer=NORM_ANSWER, max_new_tokens=MAX_NEW_TOKENS,
        overwrite=OVERWRITE,
    ))
    del vlm, editor
    gc.collect()
    torch.cuda.empty_cache()

display(Markdown(render_table(rows)))

medgemma: 50 unique edits, 1 conflicting cases dropped
  drop 1025_combined: jesse.y.liu.md, lhunterellul -> lichenoid keratosis, basosquamous cell carcinoma
medgemma: 171 locality cases


/scratch/jq2uw/conda_ex/conda_venv/dermedit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
